# CIL — Monocular Depth Estimation: ViT Baseline
Trains a ViT-Small encoder + Transformer decoder on the competition data. Runs on **Google Colab** (Kaggle data downloaded automatically).

## 0 — Config  *(edit this cell)*

In [ ]:
COMPETITION_SLUG = "ethz-cil-monocular-depth-estimation-2026"
DECODER_TYPE     = "transformer"   # 'transformer' or 'conv'
PRETRAINED       = True            # ImageNet init
EPOCHS           = 50
BATCH_SIZE       = 8
LR               = 1e-4
IMG_SIZE         = 560

## 1 — Setup

In [ ]:
import sys, os

IN_COLAB = 'google.colab' in sys.modules
print('Colab' if IN_COLAB else 'Cluster/local')

# Clone repo (idempotent)
!git clone https://github.com/ChristianDe-fabolous/CIL.git /CIL 2>/dev/null || echo 'repo already present'
%cd /CIL
!pip install -q -r requirements.txt

REPO_ROOT = '/CIL'
sys.path.insert(0, REPO_ROOT)

## 2 — Google Drive  *(saves checkpoints across sessions)*

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
CHECKPOINT_DIR = '/content/drive/MyDrive/cil_checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f'Checkpoints → {CHECKPOINT_DIR}')

## 3 — Download data

In [ ]:
import kagglehub, zipfile, glob as _glob

kagglehub.login()
data_cache = kagglehub.competition_download(COMPETITION_SLUG)
print(f'Cache: {data_cache}')

# Extract any zip files
for zf in _glob.glob(os.path.join(data_cache, '**', '*.zip'), recursive=True):
    print(f'Extracting {zf} ...')
    with zipfile.ZipFile(zf) as z:
        z.extractall(os.path.dirname(zf))
    os.remove(zf)

# Find the extracted competition directory (contains 'monodepth')
DATA_ROOT = next(
    (os.path.join(data_cache, d) for d in os.listdir(data_cache) if 'monodepth' in d.lower()),
    data_cache,
)
print(f'DATA_ROOT={DATA_ROOT}')

# Auto-detect subdirectory layout
def _find_subdir(root, candidates, ext):
    for d in candidates:
        if _glob.glob(os.path.join(root, d, f'*.{ext}')):
            return d
    raise FileNotFoundError(f'No .{ext} files found under {root} in {candidates}')

TRAIN_IMG_SUBDIR   = _find_subdir(DATA_ROOT, ['train/images', 'train'], 'png')
TRAIN_DEPTH_SUBDIR = _find_subdir(DATA_ROOT, ['train/depth',  'train'], 'npy')
TEST_IMG_SUBDIR    = _find_subdir(DATA_ROOT, ['test/images',  'test'],  'png')

n_train = len(_glob.glob(os.path.join(DATA_ROOT, TRAIN_IMG_SUBDIR, '*.png')))
n_depth = len(_glob.glob(os.path.join(DATA_ROOT, TRAIN_DEPTH_SUBDIR, '*.npy')))
n_test  = len(_glob.glob(os.path.join(DATA_ROOT, TEST_IMG_SUBDIR, '*.png')))
print(f'train images={n_train}  depth maps={n_depth}  test images={n_test}')

## 4 — Build config

In [ ]:
import yaml

cfg = {
    'model': {
        'encoder_pretrained': PRETRAINED,
        'decoder_type':       DECODER_TYPE,
        'decoder_blocks':     5,
        'embed_dim':          384,
        'num_heads':          6,
        'img_size':           IMG_SIZE,
        'patch_size':         16,
    },
    'data': {
        'data_root':       DATA_ROOT,
        'train_image_dir': TRAIN_IMG_SUBDIR,
        'train_depth_dir': TRAIN_DEPTH_SUBDIR,
        'test_image_dir':  TEST_IMG_SUBDIR,
        'num_workers':     2,
        'val_split':       0.1,
    },
    'training': {
        'epochs':       EPOCHS,
        'batch_size':   BATCH_SIZE,
        'lr':           LR,
        'weight_decay': 0.01,
        'grad_clip':    1.0,
        'amp':          True,
        'seed':         42,
    },
    'logging': {
        'log_interval':    20,
        'checkpoint_dir':  CHECKPOINT_DIR,
        'experiment_name': 'vit_depth',
    },
}

cfg_path = os.path.join(REPO_ROOT, 'configs', 'runtime_config.yaml')
with open(cfg_path, 'w') as f:
    yaml.dump(cfg, f)
print(yaml.dump(cfg))

## 5 — Train

In [ ]:
import subprocess

cmd = [sys.executable, os.path.join(REPO_ROOT, 'src', 'train.py'), '--config', cfg_path]
print('Running:', ' '.join(cmd))
result = subprocess.run(cmd, cwd=REPO_ROOT)
print('Exit code:', result.returncode)

## 6 — Predict

In [ ]:
run_name  = f"vit_depth_{DECODER_TYPE}_pretrained{PRETRAINED}"
CKPT_PATH = os.path.join(CHECKPOINT_DIR, run_name, 'best.pth')
PRED_DIR  = '/content/predictions'

assert os.path.exists(CKPT_PATH), f'Checkpoint not found: {CKPT_PATH}'

cmd = [
    sys.executable, os.path.join(REPO_ROOT, 'src', 'predict.py'),
    '--checkpoint', CKPT_PATH,
    '--test_dir',   os.path.join(DATA_ROOT, TEST_IMG_SUBDIR),
    '--output_dir', PRED_DIR,
    '--batch_size', str(BATCH_SIZE),
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=REPO_ROOT)

## 7 — Submit

In [ ]:
from pathlib import Path
import numpy as np, base64, zlib, pandas as pd

def encode_depth(depth):
    depth = np.asarray(depth, dtype=np.float16)
    return base64.b64encode(zlib.compress(depth.tobytes(), 9)).decode()

rows = []
for p in sorted(Path(PRED_DIR).glob('test_*.npy')):
    depth = np.load(p)
    idx = p.stem.split('_')[-1]
    rows.append({'id': f'test_{idx}_depth', 'Depths': encode_depth(depth)})

df = pd.DataFrame(rows, columns=['id', 'Depths'])
sub_path = '/content/submission.csv'
df.to_csv(sub_path, index=False)
print(f'Saved {len(df)} rows → {sub_path}')

from google.colab import files
files.download(sub_path)